### **Supplementary Figure S2: Stratified correlation analysis**

Addresses whether Native-IVT correlations are driven by high-abundance rRNA or hold across the transcriptome, by stratifying coverage into all-genome, rRNA-only, and non-rRNA regions. Set the data path in the cell below before running.

**Samples:** Native RNA (3 replicates), IVT RNA (3 replicates) — paired (Native 1 → IVT 1, etc.)

**Reference:** *E. coli* K-12 (ENA|U00096|U00096.3)

**rRNA coordinates:** 22 rRNA genes across 7 operons (hardcoded from GenBank annotation NC_000913.3)

### Figures in this notebook: Stratified Correlation Analysis

| Figure | Description |
|--------|-------------|
| **Fig. S2** | 3×3 hexbin grid — rows: replicates 1–3, columns: all genome / rRNA only / non-rRNA only |

All figures generated at three resolutions: per-base, 100 bp bins, 1 kb bins.

---

### Requirements

- **Python** 3.10+ with `matplotlib`, `pandas`, `numpy`, `scipy`
- **Read counts**: obtained with `samtools view -c -F 2308` (primary alignments only) — hardcoded in the read counts cell
- **rRNA coordinates**: hardcoded in the `RRNA_GENES` cell — no external coordinate file needed

### Input file layout

Place the notebook and the following files/folders in the same directory:

```
your_folder/
├── supp_fig_S2_stratified_correlations.ipynb
└── coverages/
    ├── native/
    │   ├── k12_native_full_bc1.txt
    │   ├── k12_native_full_bc2.txt
    │   └── k12_native_full_bc3.txt
    └── ivt/
        ├── k12_ivt_full_bc1.txt
        ├── k12_ivt_full_bc2.txt
        └── k12_ivt_full_bc3.txt
```

Coverage files are tab-separated with columns `chrom`, `position`, `depth` (standard `samtools depth` output). Output figures are saved to `per_base/`, `binned_100bp/`, and `binned_1kb/` subfolders.

> **Note:** `DATA_DIR` resolves to the directory Jupyter was launched from, which is normally the notebook's folder. If files fail to load, set `DATA_DIR` explicitly to an absolute path.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib.gridspec import GridSpec
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 12
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['figure.dpi'] = 300
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['ytick.major.width'] = 1.0

# set to the directory containing coverages/
DATA_DIR = Path('.').resolve()
# DATA_DIR = Path('/absolute/path/to/your/data')

## rRNA gene coordinates

*E. coli* K-12 MG1655 (NC_000913.3) — 22 rRNA genes across 7 operons, verified from GenBank annotation.

In [ ]:
RRNA_GENES = [
    # rrnH operon
    ('rrsH', 223771, 225312),
    ('rrlH', 225759, 228662),
    ('rrfH', 228756, 228875),
    # rrnG operon
    ('rrfG', 2726069, 2726188),
    ('rrlG', 2726281, 2729184),
    ('rrsG', 2729616, 2731157),
    # rrnD operon (2 × 5S)
    ('rrfF', 3423423, 3423542),
    ('rrfD', 3423668, 3423787),
    ('rrlD', 3423880, 3426783),
    ('rrsD', 3427221, 3428762),
    # rrnC operon
    ('rrsC', 3941808, 3943349),
    ('rrlC', 3943704, 3946607),
    ('rrfC', 3946700, 3946819),
    # rrnA operon
    ('rrsA', 4035531, 4037072),
    ('rrlA', 4037519, 4040423),
    ('rrfA', 4040517, 4040636),
    # rrnB operon
    ('rrsB', 4166659, 4168200),
    ('rrlB', 4168641, 4171544),
    ('rrfB', 4171637, 4171756),
    # rrnE operon
    ('rrsE', 4208147, 4209688),
    ('rrlE', 4210043, 4212946),
    ('rrfE', 4213040, 4213159),
]

print(f"rRNA genes: {len(RRNA_GENES)}, total bp: {sum(e - s + 1 for _, s, e in RRNA_GENES):,}")

## Load data

In [ ]:
def load_coverage_file(filepath):
    df = pd.read_csv(filepath, sep='\t', header=None,
                     names=['chromosome', 'position', 'depth'])
    print(f"  {filepath.name}: {len(df):,} positions")
    return df

native1 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc1.txt')
native2 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc2.txt')
native3 = load_coverage_file(DATA_DIR / 'coverages/native/k12_native_full_bc3.txt')

ivt1 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc1.txt')
ivt2 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc2.txt')
ivt3 = load_coverage_file(DATA_DIR / 'coverages/ivt/k12_ivt_full_bc3.txt')

## CPM normalisation

In [ ]:
# read counts from samtools view -c -F 2308
read_counts = {
    'native1': 88366,
    'native2': 235984,
    'native3': 347917,
    'ivt1': 72626,
    'ivt2': 50142,
    'ivt3': 49260
}

def calculate_cpm(df, total_reads):
    df_copy = df.copy()
    df_copy['cpm'] = (df_copy['depth'] / total_reads) * 1_000_000
    return df_copy

native1 = calculate_cpm(native1, read_counts['native1'])
native2 = calculate_cpm(native2, read_counts['native2'])
native3 = calculate_cpm(native3, read_counts['native3'])
ivt1 = calculate_cpm(ivt1, read_counts['ivt1'])
ivt2 = calculate_cpm(ivt2, read_counts['ivt2'])
ivt3 = calculate_cpm(ivt3, read_counts['ivt3'])

## Label positions by region type

In [ ]:
def is_rrna_position(position, rrna_genes):
    for gene, start, end in rrna_genes:
        if start <= position <= end:
            return True
    return False

print("Labelling positions (may take 15-30 s per sample)...")
for sample_name, sample_df in [
    ('Native1', native1), ('Native2', native2), ('Native3', native3),
    ('IVT1', ivt1), ('IVT2', ivt2), ('IVT3', ivt3)
]:
    sample_df['region_type'] = sample_df['position'].apply(
        lambda pos: 'rRNA' if is_rrna_position(pos, RRNA_GENES) else 'non-rRNA'
    )
    n_rrna = (sample_df['region_type'] == 'rRNA').sum()
    n_non = (sample_df['region_type'] == 'non-rRNA').sum()
    print(f"  {sample_name}: rRNA {n_rrna:,} ({100*n_rrna/len(sample_df):.2f}%), "
          f"non-rRNA {n_non:,} ({100*n_non/len(sample_df):.2f}%)")

## Binning (preserves region type)

In [ ]:
def bin_coverage_with_region(df, bin_size):
    binned = df.copy()
    binned['bin_start'] = ((binned['position'] - 1) // bin_size) * bin_size + 1
    agg = binned.groupby('bin_start').agg(
        cpm=('cpm', 'mean'),
        region_type=('region_type', lambda x: x.mode().iloc[0])  # majority vote
    ).reset_index()
    agg['position'] = agg['bin_start'] + (bin_size - 1) / 2.0
    return agg


def merge_binned_samples_with_region(df1, df2, bin_size, name1, name2):
    df1_binned = bin_coverage_with_region(df1, bin_size)
    df2_binned = bin_coverage_with_region(df2, bin_size)
    merged = pd.merge(
        df1_binned[['position', 'cpm', 'region_type']],
        df2_binned[['position', 'cpm', 'region_type']],
        on=['position', 'region_type'],
        suffixes=(f'_{name1}', f'_{name2}')
    )
    print(f"  {name1} vs {name2} ({bin_size}bp bins): {len(merged):,} bins")
    return merged

## Merge paired samples

In [ ]:
def merge_two_samples(df1, df2, name1, name2):
    merged = pd.merge(
        df1[['position', 'cpm', 'region_type']],
        df2[['position', 'cpm', 'region_type']],
        on=['position', 'region_type'],
        suffixes=(f'_{name1}', f'_{name2}')
    )
    print(f"  {name1} vs {name2}: {len(merged):,} positions")
    return merged

# per-base
native1_ivt1 = merge_two_samples(native1, ivt1, 'Native1', 'IVT1')
native2_ivt2 = merge_two_samples(native2, ivt2, 'Native2', 'IVT2')
native3_ivt3 = merge_two_samples(native3, ivt3, 'Native3', 'IVT3')

# 100 bp bins
print("\n100bp bins:")
native1_ivt1_100bp = merge_binned_samples_with_region(native1, ivt1, 100, 'Native1', 'IVT1')
native2_ivt2_100bp = merge_binned_samples_with_region(native2, ivt2, 100, 'Native2', 'IVT2')
native3_ivt3_100bp = merge_binned_samples_with_region(native3, ivt3, 100, 'Native3', 'IVT3')

# 1 kb bins
print("\n1kb bins:")
native1_ivt1_1kb = merge_binned_samples_with_region(native1, ivt1, 1000, 'Native1', 'IVT1')
native2_ivt2_1kb = merge_binned_samples_with_region(native2, ivt2, 1000, 'Native2', 'IVT2')
native3_ivt3_1kb = merge_binned_samples_with_region(native3, ivt3, 1000, 'Native3', 'IVT3')

## Stratified correlations

In [ ]:
def calculate_stratified_correlations(merged_df, name1, name2):
    results = {}
    categories = {
        'All genome': merged_df,
        'rRNA only': merged_df[merged_df['region_type'] == 'rRNA'],
        'Non-rRNA only': merged_df[merged_df['region_type'] == 'non-rRNA']
    }
    for category, df_subset in categories.items():
        cpm1 = df_subset[f'cpm_{name1}'].values
        cpm2 = df_subset[f'cpm_{name2}'].values
        mask = (cpm1 > 0) & (cpm2 > 0)
        cpm1_filtered = cpm1[mask]
        cpm2_filtered = cpm2[mask]
        if len(cpm1_filtered) < 100:
            print(f"  WARNING: Only {len(cpm1_filtered)} positions in {category}")
            results[category] = None
            continue

        # log transform for Pearson — consistent with what the hexbin axes show
        cpm1_log = np.log10(cpm1_filtered + 1)
        cpm2_log = np.log10(cpm2_filtered + 1)

        spearman_r, spearman_p = stats.spearmanr(cpm1_filtered, cpm2_filtered)
        pearson_r, pearson_p = stats.pearsonr(cpm1_log, cpm2_log)

        results[category] = {
            'n_total': len(df_subset),
            'n_with_coverage': len(cpm1_filtered),
            'pct_covered': 100 * len(cpm1_filtered) / len(df_subset),
            'spearman_r': spearman_r,
            'spearman_p': spearman_p,
            'pearson_r': pearson_r,
            'pearson_p': pearson_p,
            'cpm1_filtered': cpm1_filtered,
            'cpm2_filtered': cpm2_filtered
        }
        print(f"  {category:15s}: n={len(cpm1_filtered):,}, \u03c1={spearman_r:.3f}, r={pearson_r:.3f}")
    return results

print("Per-base — Replicate 1:")
stats_rep1 = calculate_stratified_correlations(native1_ivt1, 'Native1', 'IVT1')
print("\nPer-base — Replicate 2:")
stats_rep2 = calculate_stratified_correlations(native2_ivt2, 'Native2', 'IVT2')
print("\nPer-base — Replicate 3:")
stats_rep3 = calculate_stratified_correlations(native3_ivt3, 'Native3', 'IVT3')

print("\n100bp bins — Replicate 1:")
stats_rep1_100bp = calculate_stratified_correlations(native1_ivt1_100bp, 'Native1', 'IVT1')
print("\n100bp bins — Replicate 2:")
stats_rep2_100bp = calculate_stratified_correlations(native2_ivt2_100bp, 'Native2', 'IVT2')
print("\n100bp bins — Replicate 3:")
stats_rep3_100bp = calculate_stratified_correlations(native3_ivt3_100bp, 'Native3', 'IVT3')

print("\n1kb bins — Replicate 1:")
stats_rep1_1kb = calculate_stratified_correlations(native1_ivt1_1kb, 'Native1', 'IVT1')
print("\n1kb bins — Replicate 2:")
stats_rep2_1kb = calculate_stratified_correlations(native2_ivt2_1kb, 'Native2', 'IVT2')
print("\n1kb bins — Replicate 3:")
stats_rep3_1kb = calculate_stratified_correlations(native3_ivt3_1kb, 'Native3', 'IVT3')

In [ ]:
datasets = {
    'per_base': {
        'merged_data': [
            (native1_ivt1, 'Native1', 'IVT1'),
            (native2_ivt2, 'Native2', 'IVT2'),
            (native3_ivt3, 'Native3', 'IVT3')
        ],
        'stats': [stats_rep1, stats_rep2, stats_rep3],
        'suffix': '',
        'label': 'Per-base'
    },
    'binned_100bp': {
        'merged_data': [
            (native1_ivt1_100bp, 'Native1', 'IVT1'),
            (native2_ivt2_100bp, 'Native2', 'IVT2'),
            (native3_ivt3_100bp, 'Native3', 'IVT3')
        ],
        'stats': [stats_rep1_100bp, stats_rep2_100bp, stats_rep3_100bp],
        'suffix': '_100bp',
        'label': '100bp bins'
    },
    'binned_1kb': {
        'merged_data': [
            (native1_ivt1_1kb, 'Native1', 'IVT1'),
            (native2_ivt2_1kb, 'Native2', 'IVT2'),
            (native3_ivt3_1kb, 'Native3', 'IVT3')
        ],
        'stats': [stats_rep1_1kb, stats_rep2_1kb, stats_rep3_1kb],
        'suffix': '_1kb',
        'label': '1kb bins'
    }
}

## Summary table

In [ ]:
all_stats = []
for resolution, config in datasets.items():
    for rep_name, stats_dict in [('Rep1', config['stats'][0]),
                                  ('Rep2', config['stats'][1]),
                                  ('Rep3', config['stats'][2])]:
        for category in ['All genome', 'rRNA only', 'Non-rRNA only']:
            if stats_dict[category] is not None:
                row = {
                    'Resolution': config['label'],
                    'Replicate': rep_name,
                    'Category': category,
                    'n_total': stats_dict[category]['n_total'],
                    'n_with_coverage': stats_dict[category]['n_with_coverage'],
                    'pct_covered': stats_dict[category]['pct_covered'],
                    'spearman_r': stats_dict[category]['spearman_r'],
                    'spearman_p': stats_dict[category]['spearman_p'],
                    'pearson_r': stats_dict[category]['pearson_r'],
                    'pearson_p': stats_dict[category]['pearson_p']
                }
                all_stats.append(row)

summary_df = pd.DataFrame(all_stats)
summary_df = summary_df[['Resolution', 'Replicate', 'Category', 'n_total', 'n_with_coverage',
                         'pct_covered', 'spearman_r', 'spearman_p', 'pearson_r', 'pearson_p']]
summary_df['pct_covered'] = summary_df['pct_covered'].round(2)
summary_df['spearman_r'] = summary_df['spearman_r'].round(3)
summary_df['pearson_r'] = summary_df['pearson_r'].round(3)

summary_df.to_csv('SupplementaryTable_S3_Stratified_Correlations_all_resolutions.csv', index=False)
print(summary_df[['Resolution', 'Replicate', 'Category', 'n_with_coverage', 'spearman_r', 'pearson_r']].to_string(index=False))

print("\nAverages by resolution and category:")
for resolution in ['Per-base', '100bp bins', '1kb bins']:
    print(f"  {resolution}:")
    res_df = summary_df[summary_df['Resolution'] == resolution]
    for category in ['All genome', 'rRNA only', 'Non-rRNA only']:
        cat_data = res_df[res_df['Category'] == category]
        if len(cat_data) > 0:
            avg = cat_data['spearman_r'].mean()
            std = cat_data['spearman_r'].std()
            print(f"    {category:15s}: \u03c1 = {avg:.3f} \u00b1 {std:.3f}")

## Plots

In [ ]:
def format_number(n):
    if n >= 1000:
        return f"{n/1000:.0f}K"
    return str(n)


def plot_panel(ax, cpm1_log, cpm2_log, stats, panel_label, show_ylabel, show_xlabel):
    hexbin = ax.hexbin(cpm1_log, cpm2_log, gridsize=40, cmap='Purples',
                       mincnt=1, bins='log', linewidths=0.2)

    max_val = max(cpm1_log.max(), cpm2_log.max())
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.2, alpha=0.6, zorder=5)

    if show_xlabel:
        ax.set_xlabel('Native RNA coverage [log\u2081\u2080(CPM + 1)]', fontsize=18)
    if show_ylabel:
        ax.set_ylabel('IVT RNA coverage [log\u2081\u2080(CPM + 1)]', fontsize=18)

    ax.tick_params(axis='both', labelsize=16)

    stats_text = (
        f"n = {format_number(stats['n_with_coverage'])}\n"
        f"\u03c1 = {stats['spearman_r']:.3f}\n"
        f"r = {stats['pearson_r']:.3f}"
    )
    ax.text(0.97, 0.03, stats_text, transform=ax.transAxes,
            fontsize=14, va='bottom', ha='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='wheat',
                     alpha=0.9, edgecolor='gray', linewidth=0.5))

    #ax.text(-0.15, 1.08, panel_label, transform=ax.transAxes,
    #        fontsize=14, fontweight='bold', va='top', ha='right')

    ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')

    return hexbin

In [ ]:
categories = ['All genome', 'rRNA only', 'Non-rRNA only']
panel_labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i']

for resolution, config in datasets.items():
    out_dir = Path('suppfig_s2') / resolution
    out_dir.mkdir(parents=True, exist_ok=True)

    suffix = config['suffix']
    label = config['label']
    stats_list = config['stats']

    replicates = [
        ('Replicate 1', stats_list[0]),
        ('Replicate 2', stats_list[1]),
        ('Replicate 3', stats_list[2])
    ]

    fig = plt.figure(figsize=(18, 16))
    gs = GridSpec(3, 4, figure=fig,
                  hspace=0.20, wspace=0.35,
                  width_ratios=[1, 1, 1, 0.05],
                  left=0.08, right=0.95, top=0.94, bottom=0.06)

    panel_idx = 0
    hexbin_objects = []

    for row_idx, (rep_name, stats_dict) in enumerate(replicates):
        for col_idx, cat_key in enumerate(categories):
            ax = fig.add_subplot(gs[row_idx, col_idx])
            s = stats_dict[cat_key]
            if s is None:
                ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', fontsize=12)
                ax.set_xlim([0, 1])
                ax.set_ylim([0, 1])
                continue
            cpm1_log = np.log10(s['cpm1_filtered'] + 1)
            cpm2_log = np.log10(s['cpm2_filtered'] + 1)
            show_ylabel = (col_idx == 0)
            show_xlabel = (row_idx == 2)
            hexbin = plot_panel(ax, cpm1_log, cpm2_log, s,
                               panel_labels[panel_idx], show_ylabel, show_xlabel)
            hexbin_objects.append(hexbin)
            panel_idx += 1
            if row_idx == 0:
                ax.set_title(cat_key, fontsize=16, fontweight='bold', pad=10)
            if col_idx == 0:
                ax.text(-0.32, 0.5, rep_name, transform=ax.transAxes,
                       fontsize=18, fontweight='bold', va='center', ha='center', rotation=90)

    if hexbin_objects:
        cbar_ax = fig.add_subplot(gs[:, 3])
        cbar = plt.colorbar(hexbin_objects[0], cax=cbar_ax)
        cbar.set_label('Position count (log scale)', fontsize=11, labelpad=10)
        cbar.ax.tick_params(labelsize=14)

    fig.suptitle('', fontsize=14, fontweight='bold', y=0.98)

    plt.savefig(out_dir / f'SupplementaryFigure_S2_Stratified_Correlations_3x3{suffix}.png',
                dpi=300, bbox_inches='tight')
    plt.savefig(out_dir / f'SupplementaryFigure_S2_Stratified_Correlations_3x3{suffix}.pdf',
                bbox_inches='tight')
    print(f"S2 ({label}): {out_dir}/SupplementaryFigure_S2_Stratified_Correlations_3x3{suffix}.png/pdf")
    plt.close()

print("\nAll plots generated.")